### **PhantomProof** 
#### Exploratory data Analysis

THis notebook performs a structured EDA on [`events_chaotic.csv`] (events_chaotic.csv) which is the output of the chaos injection engine.

The goal is to detect, quantify, and document every failure mode injected in [`03_chaos.ipynb`] notebook so that the silver layer cleaning logic can be written efficiently and accurately. 

EDA is organised into 5 sections which are the following:
1. Dataset overview and schema check.
2. Temporal anomalies 
3. Structural anomalies
4. Operational anomalies
5. Market and integrity anomalies




In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import date

### for consistent plot style in the notebook:
sns.set_theme(style= "darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 4)

## load the chaotic events
df = pd.read_csv("events_chaotic.csv", parse_dates = ["event_timestamp"])

## load dimension tables for joins
df_node = pd.read_csv("dim_node.csv")
df_product = pd.read_csv("dim_product.csv")
df_scanner = pd.read_csv("dim_scanner.csv")
df_baseline = pd.read_csv("events_baseline.csv", parse_dates = ["event_timestamp"])

print(f"chaotic events: {len(df):,}")
print(f"Baseline events: {len(df_baseline):,}")
print(f"Net delta: {len(df) - len(df_baseline):,} (+150 dupes injected, -169 removed by blackout/masking)")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['event_timestamp'].min()} to {df['event_timestamp'].max()}")


chaotic events: 436,124
Baseline events: 436,143
Net delta: -19 (+150 dupes injected, -169 removed by blackout/masking)
Columns: ['event_id', 'event_type', 'event_timestamp', 'node_id', 'device_id', 'product_id', 'units_sold', 'units_rto', 'units_transferred', 'source_node', 'destination_node', 'stock_on_hand', 'rto_reason', 'processing_lag_hours', 'session_id']
Date range: 2025-12-01 00:00:00 to 2026-08-13 13:58:00.000


#### **Section 1**
#### **Dataset Overview and Schema check**

In [4]:
## dtypes
print("===Column datatypes===")
print(df.dtypes)
print()

## ---null profile
# units_sold is object was schema poisoning (contains strings like "9 units")
# processing_lag_hours is object after decimal separator chaos (eg: "1,23" instead of 1.23)
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_profile = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_profile = null_profile[null_profile["null_count"] > 0]. sort_values("null_count", ascending= False) # to filter and only get the columns with nulls 

print("=== Null Profile (columns with any nulls) ===")
print(null_profile.to_string())

## expected nulls are:
# rto_reason : None for non-RTO events (by design) +20 injected chaos nulls
# product_id : None for snapshot events (snapshots capture total stock, not per product)
# source_node: None for non_transfer events
# destantion_node : None for non transfer events


===Column datatypes===
event_id                object
event_type              object
event_timestamp         object
node_id                 object
device_id               object
product_id              object
units_sold              object
units_rto                int64
units_transferred        int64
source_node             object
destination_node        object
stock_on_hand            int64
rto_reason              object
processing_lag_hours    object
session_id              object
dtype: object

=== Null Profile (columns with any nulls) ===
                  null_count  null_pct
rto_reason            424299     97.29
source_node           364036     83.47
destination_node      364036     83.47
product_id             81356     18.65
